# TXT Loader (2026년 최신 권장 사용법)

`.txt` 확장자를 가지는 파일을 `Document` 로 로드하는 방법을 살펴보겠습니다.

> **⚠️ 2026년 9월 기준 변경 사항 (공통)**
>
> - 책에서 사용한 `langchain_community.document_loaders` 는 **`langchain-community` 패키지 sunset**(2026년 5월 발표, 6월 저장소 아카이브)으로 더 이상 유지보수되지 않습니다. 설치·import 는 여전히 되지만 새 프로젝트의 기반으로 권장되지 않습니다.
> - LangChain 의 현재 방향은 ① **전용 통합 패키지**(`langchain-upstage`, `langchain-unstructured`, `langchain-pymupdf4llm`, `langchain-docling` 등)를 쓰거나, ② 파싱 라이브러리를 **직접 사용**하고 결과를 `langchain_core.documents.Document` 로 감싸는 것(필요하면 `BaseLoader` 를 상속한 작은 로더 클래스 작성)입니다.
> - `Document`, `BaseLoader`, 텍스트 분할기(`langchain-text-splitters`)는 그대로 유지되는 핵심 인터페이스입니다.

**이 노트북에서 바뀐 점**

| 책(구버전, `langchain_community`) | 현재 권장 |
|---|---|
| `TextLoader(path, autodetect_encoding=True)` | `pathlib` 로 읽고, 실패 시 **`charset-normalizer`** 로 인코딩 자동 감지 |
| `DirectoryLoader(..., loader_cls=TextLoader, silent_errors=True)` | `Path.glob()` + `try/except` + `logging` (디렉토리 로딩은 `11-Directory-Loader` 참고) |

In [ ]:
# 설치
# !pip install -qU langchain-core charset-normalizer

In [ ]:
import logging
from pathlib import Path
from typing import Iterator

from charset_normalizer import from_path
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


class TextFileLoader(BaseLoader):
    """텍스트 파일 1개 → Document 1개 (인코딩 자동 감지 지원)"""

    def __init__(
        self, file_path: str | Path, encoding: str = "utf-8", autodetect_encoding: bool = False
    ) -> None:
        self.file_path = Path(file_path)
        self.encoding = encoding
        self.autodetect_encoding = autodetect_encoding

    def lazy_load(self) -> Iterator[Document]:
        try:
            text, used = self.file_path.read_text(encoding=self.encoding), self.encoding
        except UnicodeDecodeError:
            if not self.autodetect_encoding:
                raise
            best = from_path(self.file_path).best()  # 가장 그럴듯한 인코딩 추정
            if best is None:
                raise RuntimeError(f"인코딩을 감지하지 못했습니다: {self.file_path}")
            text, used = str(best), best.encoding
        yield Document(page_content=text, metadata={"source": str(self.file_path), "encoding": used})

In [ ]:
# 텍스트 로더 생성
loader = TextFileLoader("data/appendix-keywords.txt")

# 문서 로드
docs = loader.load()
print(f"문서의 수: {len(docs)}\n")
print("[메타데이터]\n")
print(docs[0].metadata)
print("\n========= [앞부분] 미리보기 =========\n")
print(docs[0].page_content[:500])

## 파일 인코딩 자동 감지

디렉토리에서 인코딩이 제각각인 여러 텍스트 파일을 한꺼번에 로드할 때 유용한 전략입니다.

- **오류 무시(구 `silent_errors`)**: 로드할 수 없는 파일은 경고를 남기고 건너뜁니다.
- **자동 감지(구 `autodetect_encoding`)**: UTF-8 로 읽기에 실패하면 `charset-normalizer` 로 인코딩을 추정합니다.

In [ ]:
logger = logging.getLogger(__name__)


def load_text_files(path: str, glob: str = "**/*.txt", silent_errors: bool = True, **loader_kwargs):
    docs = []
    for file in sorted(Path(path).glob(glob)):
        if not file.is_file():
            continue
        try:
            docs.extend(TextFileLoader(file, **loader_kwargs).load())
        except Exception as e:
            if not silent_errors:
                raise
            logger.warning("건너뜀: %s (%s)", file, e)
    return docs


docs = load_text_files("data/", glob="**/*.txt", silent_errors=True, autodetect_encoding=True)

`data/appendix-keywords.txt` 파일과 파일명이 유사한 파생 파일들은 모두 인코딩 방식이 다른 파일들입니다. 메타데이터의 `encoding` 에서 감지된 인코딩을 확인할 수 있습니다.

In [ ]:
[(doc.metadata["source"], doc.metadata["encoding"]) for doc in docs]

In [ ]:
print("[메타데이터]\n")
print(docs[2].metadata)
print("\n========= [앞부분] 미리보기 =========\n")
print(docs[2].page_content[:500])

In [ ]:
print("[메타데이터]\n")
print(docs[3].metadata)
print("\n========= [앞부분] 미리보기 =========\n")
print(docs[3].page_content[:500])

In [ ]:
print("[메타데이터]\n")
print(docs[4].metadata)
print("\n========= [앞부분] 미리보기 =========\n")
print(docs[4].page_content[:500])